# HackCafe Vision Baseline\n\nNotebook Colab para iterar modelos de visao computacional do HackCafe. A primeira trilha usa o BRACOL local como baseline de classificacao de estresse predominante; trilhas opcionais usam datasets externos para deteccao YOLO quando houver credenciais Kaggle/Roboflow.\n\nDecisao de produto: preservar o frontend atual do HackCafe e evoluir a inteligencia visual por contratos e artefatos versionados.

In [ ]:
!nvidia-smi || true\n!pip -q install ultralytics pandas scikit-learn pillow tqdm

In [ ]:
from pathlib import Path\nimport json, os, subprocess\n\nREPO_URL = 'https://github.com/C-Icaro/HackCafe.git'\nBRANCH = 'ml-public-cv-datasets'\nWORKDIR = Path('/content/HackCafe')\n\nif not WORKDIR.exists():\n    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORKDIR)], check=True)\nos.chdir(WORKDIR)\nprint('Workspace:', Path.cwd())

In [ ]:
manifest = json.loads(Path('análise-preditiva/dataset_manifest.json').read_text(encoding='utf-8'))\nprint('Primary dataset:', manifest['decision']['primary_dataset'])\nfor ds in manifest['datasets']:\n    print(f"{ds['tier']:>2} | {ds['fit_score']:>3} | {ds['id']} | {', '.join(ds['tasks'][:2])}")

## Preparar BRACOL para classificacao\n\nO script cria a estrutura `train/`, `val/`, `test/` esperada por classificadores Ultralytics. Por padrao, a classe `5` do CSV e excluida como inconclusiva/mista para evitar ruido na primeira baseline. O checkout atual pode estar incompleto em relacao ao BRACOL original; use o campo `missing_images` do resumo como evidencia e baixe a fonte Mendeley completa antes de um treino de producao.

In [ ]:
!python análise-preditiva/prepare_bracol_classification.py --output /content/hackcafe_runs/bracol_cls --mode copy

## Treinar baseline\n\nComece pequeno para validar o pipeline. Em Colab Pro/Pro+, aumente `epochs`, troque `MODEL_NAME` para uma variante maior, ou rode uma grade de modelos. Se a versao instalada do Ultralytics suportar familia mais nova, altere o nome do peso aqui.

In [ ]:
from ultralytics import YOLO\n\nMODEL_NAME = 'yolo11s-cls.pt'\nDATA_DIR = '/content/hackcafe_runs/bracol_cls'\n\nmodel = YOLO(MODEL_NAME)\nresults = model.train(\n    data=DATA_DIR,\n    epochs=30,\n    imgsz=224,\n    batch=-1,\n    patience=8,\n    seed=42,\n    project='/content/hackcafe_runs',\n    name='bracol_cls_baseline',\n    exist_ok=True,\n)

In [ ]:
best = Path('/content/hackcafe_runs/bracol_cls_baseline/weights/best.pt')\ntrained = YOLO(str(best))\nmetrics = trained.val(data=DATA_DIR, split='test', imgsz=224)\nprint(metrics)

## Opcional: datasets externos\n\n- Kaggle: configure `KAGGLE_USERNAME` e `KAGGLE_KEY` como secrets do Colab antes de baixar BRACOL-YOLO ou Coffee Fruit Maturity.\n- Roboflow: configure `ROBOFLOW_API_KEY` como secret do Colab antes de exportar datasets do Universe.\n- Devin Cloud: usar o prompt de `documentos/devin-orquestracao.md` e pedir benchmark assinado com SHA, metricas e artefatos.

In [ ]:
# Exemplo Kaggle, execute somente apos configurar secrets no Colab.\n# !pip -q install kaggle\n# !mkdir -p ~/.kaggle\n# !printf '{"username":"%s","key":"%s"}' "$KAGGLE_USERNAME" "$KAGGLE_KEY" > ~/.kaggle/kaggle.json\n# !chmod 600 ~/.kaggle/kaggle.json\n# !kaggle datasets download -d jonatanfragoso/bracol-for-yolov8-detection -p /content/datasets/bracol-yolo --unzip\n# !yolo detect train model=yolo11s.pt data=/content/datasets/bracol-yolo/data.yaml epochs=50 imgsz=640 project=/content/hackcafe_runs name=bracol_yolo_detect

In [ ]:
# Export para integracao futura.\n# trained.export(format='onnx')\n# trained.export(format='torchscript')